# Worked examples for the strategy cards


Each strategy card shows one real failure and one real success. This notebook chooses them from the calibration sweep on disk and, for each success, runs the strategy once to list what it says about genes without a known label. The code is `scripts/build_strategy_examples.py`; this notebook calls it, so what is shown is what ships in `starplast/data/strategy_examples.json`.

Nothing here changes a calibration number: the failure and the success are records already in `runs.jsonl`, and the explanations are written from their own numbers.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '/media/carruthers/mnt3/claude/repo/starplast/.claude/worktrees/agent-a49e8967eec84690a'); sys.path.insert(0, '/media/carruthers/mnt3/claude/repo/starplast/.claude/worktrees/agent-a49e8967eec84690a/scripts')
import build_strategy_examples as B
runs = B.load_runs()
print(len(runs), 'runs;', runs['verdict'].value_counts().to_dict())

7640 runs; {'PASS': 4794, 'FAIL': 2399, 'INCONCLUSIVE': 417, 'NOT RUN': 30}


## 1. The choice rule, strategy by strategy

Success: the PASS at the default setting with the highest skill. Failure: the FAIL at the default setting with the lowest skill, else any FAIL, else the self-test on the noise table.

In [2]:
from starplast import strategies as S
rows = []
for org in ('Tg', 'Pf'):
    for s in S.catalog():
        if None and s.key not in None: continue
        c = B.choose(runs, s.key, org)
        rows.append({'organism': org, 'strategy': s.key, 'runs': c['runs'], 'passes': c['passes'], 'fails': c['fails'], 'success_at_default': c['success_at_default'], 'failure_at_default': c['failure_at_default']})
import pandas as pd
choice = pd.DataFrame(rows)
choice

,organism,strategy,runs,passes,fails,success_at_default,failure_at_default
0,Tg,holdout_search,150,71,79,True,True
1,Tg,geneset_hunt,100,18,82,True,True
2,Tg,recoverability_atlas,75,60,15,True,True
3,Tg,consensus_modules,75,37,38,True,True
4,Tg,blind_battery,70,70,0,True,False
5,Tg,block_ablation,75,50,25,True,True
6,Tg,feature_knn,225,139,86,True,True
7,Tg,map_neighbours,225,131,94,True,True
8,Tg,cluster_guilt,300,150,9,True,False
9,Tg,label_outliers,75,74,1,True,False


## 2. Build every example

Each success is run once on the shipped table; a run longer than 300 seconds is stopped and recorded as skipped.

In [3]:
data = B.build(('Tg', 'Pf'), only=None, run_new=True)
table = B.summary(data)
table

Tg 01 holdout_search: failure calibration FAIL, success PASS (46s)
Tg 02 geneset_hunt: failure calibration FAIL, success PASS (35s)
Tg 03 recoverability_atlas: failure calibration FAIL, success PASS (8s)
Tg 04 consensus_modules: failure calibration FAIL, success PASS (52s)
Tg 05 blind_battery: failure noise table INCONCLUSIVE, success PASS (14s)
Tg 06 block_ablation: failure calibration FAIL, success PASS (7s)
Tg 07 feature_knn: failure calibration FAIL, success PASS (0s)
Tg 08 map_neighbours: failure calibration FAIL, success PASS (13s)
Tg 09 cluster_guilt: failure calibration FAIL, success PASS (0s)
Tg 10 label_outliers: failure calibration FAIL, success PASS (1s)
Tg 11 layer_propagation: failure calibration FAIL, success PASS (1s)
Tg 12 layer_vote: failure calibration FAIL, success PASS (2s)
Tg 13 physical_partners: failure calibration FAIL, success PASS (2s)
Tg 14 structural_homology: failure noise table PASS, success PASS (1s)
Tg 15 multiplex_modules: failure calibration FAIL, suc

,organism,strategy,task,failure_from,failure_verdict,failure_target,failure_skill,success_target,success_skill,new_rows,new_skipped
0,Tg,holdout_search,cluster recovery,calibration,FAIL,screen_any_phenotype,0.003,compartment,0.0821,5,
1,Tg,geneset_hunt,set retrieval,calibration,FAIL,tachyzoite (stage_enriched_derived),-0.0097,PM - integral (compartment),0.0606,5,
2,Tg,recoverability_atlas,ranking,calibration,FAIL,screen_any_phenotype,-0.0632,compartment,0.7322,5,
3,Tg,consensus_modules,cluster recovery,calibration,FAIL,screen_any_phenotype,-0.0303,compartment,0.1096,5,
4,Tg,blind_battery,replication,noise table,INCONCLUSIVE,,NaN,transcription,1,5,
5,Tg,block_ablation,label calls,calibration,FAIL,dtm_class,0.0396,compartment,0.1392,5,
6,Tg,feature_knn,label calls,calibration,FAIL,screen_any_phenotype,-0.0156,compartment,0.3414,5,
7,Tg,map_neighbours,label calls,calibration,FAIL,dtm_class,-0.0081,compartment,0.2255,5,
8,Tg,cluster_guilt,label calls,calibration,FAIL,lopit_unified,0.0502,compartment,0.3178,5,
9,Tg,label_outliers,ranking,calibration,FAIL,screen_any_phenotype,0.189,compartment,0.6536,5,


## 3. Where the failures came from

A real-data failure where the sweep has one; the noise table where it has none.

In [4]:
table.groupby(['organism', 'failure_from']).size()

organism  failure_from
Pf        calibration     24
          noise table     15
Tg        calibration     30
          noise table      9

## 4. Two examples in full

In [5]:
import json
org = next(o for o in data if o != 'meta')
for k in list(data[org])[:2]:
    e = data[org][k]
    print(org, k)
    print('FAILS WHEN:', e['failure']['explanation'])
    print('WORKS WHEN:', e['success']['explanation'])
    print(json.dumps((e['success'].get('new') or {}).get('rows'), indent=1))

Tg holdout_search
FAILS WHEN: 'screen_any_phenotype' is not encoded in what this strategy reads: F1 of hidden genes in the cluster chosen for their label on known genes was 0.684 against 0.683 on shuffled data (skill 0.00).
WORKS WHEN: F1 of hidden genes in the cluster chosen for their label on known genes reached 0.12 against 0.04 on shuffled data (skill 0.08, 577 scored). Run once on every gene, it called 63 genes with no known compartment; the top 5 are listed.
[
 {
  "gene_id": "TGME49_253870",
  "product": "hypothetical protein",
  "call": "nucleus - chromatin",
  "support": "support 0.388"
 },
 {
  "gene_id": "TGME49_270720",
  "product": "hypothetical protein",
  "call": "nucleus - chromatin",
  "support": "support 0.388"
 },
 {
  "gene_id": "TGME49_242055",
  "product": "DEAD/DEAH box helicase domain-containing protein",
  "call": "nucleus - chromatin",
  "support": "support 0.388"
 },
 {
  "gene_id": "TGME49_242415",
  "product": "histone lysine-specific demethylase",
  "call"

## 5. Every explanation

The sentence each card shows under its two mini scorecards, for every strategy.

In [6]:
for o in data:
    if o == 'meta': continue
    for k, e in data[o].items():
        print(f'{o} {k}\n  FAILS WHEN: {e["failure"]["explanation"]}'
              f'\n  WORKS WHEN: {e["success"]["explanation"]}')

Tg holdout_search
  FAILS WHEN: 'screen_any_phenotype' is not encoded in what this strategy reads: F1 of hidden genes in the cluster chosen for their label on known genes was 0.684 against 0.683 on shuffled data (skill 0.00).
  WORKS WHEN: F1 of hidden genes in the cluster chosen for their label on known genes reached 0.12 against 0.04 on shuffled data (skill 0.08, 577 scored). Run once on every gene, it called 63 genes with no known compartment; the top 5 are listed.
Tg geneset_hunt
  FAILS WHEN: 'tachyzoite (stage_enriched_derived)' is not encoded in what this strategy reads: F1 of the hidden members against the best cluster's other genes was 0.038 against 0.047 on shuffled data (skill -0.01). Also, the set was too wide: 890 genes returned, only 2% of them members.
  WORKS WHEN: F1 of the hidden members against the best cluster's other genes reached 0.08 against 0.02 on shuffled data (skill 0.06, 40 scored). Run once on every gene, it placed 85 genes with no place on the list in the 

## 6. Write the shipped file

In [7]:
B.write(data)

'/media/carruthers/mnt3/claude/repo/starplast/.claude/worktrees/agent-a49e8967eec84690a/starplast/data/strategy_examples.json'